# Train Shared Vision Backbone (Ball + Marker CNN)
This notebook clones the repository, extracts the merged `shared_vision` gold dataset from your Google Drive, and trains the ~70K-param Shared Encoder Backbone (ball regression head + marker segmentation head + marker heatmap head) defined in `train_cnn_2d_tracker_marker.py`.

In [ ]:
DATASET_NAME = 'shared_vision'
VERSION = 'v1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf /content/ball_balance_video_controlled
!git clone https://github.com/Jack0468/ball_balance_video_controlled.git
!pip install pandas torch torchvision albumentations opencv-python-headless matplotlib
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Unzip the merged 03_gold shared_vision dataset (images/ + masks/ + labels.csv)
!mkdir -p /content/ball_balance_video_controlled/host_software/data/03_gold
!unzip -q -o /content/drive/MyDrive/{DATASET_NAME}.zip -d /content/ball_balance_video_controlled/host_software/data/03_gold
print("Dataset unzipped!")

### Training

In [ ]:
%cd /content/ball_balance_video_controlled
# Run as a module (-m), not a plain script -- the trainer imports
# host_software.ml_vision.training.{shared_vision_dataset,augmentations} as package-qualified
# paths, which only resolve when the repo root is on sys.path (i.e. invoked via -m from here).
!python -m host_software.ml_vision.training.train_cnn_2d_tracker_marker \
    --csv-file host_software/data/03_gold/{DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{DATASET_NAME}/masks \
    --output-dir /content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}

### Training Results
`train_cnn_2d_tracker_marker.py` exports the best checkpoint to ONNX and writes a per-session validation breakdown and loss curve.

In [ ]:
from IPython.display import Image, display
import pandas as pd

output_dir = f'/content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}'
display(Image(filename=f'{output_dir}/training_curve.png'))
pd.read_csv(f'{output_dir}/per_session_eval.csv')

### Evaluation
Runs `evaluate_shared_vision_backbone.py` against the same held-out temporal validation split used at training time (same `--val-fraction`), and reports ball-position pixel error, marker mask IoU/Dice, heatmap MSE, and inference latency -- metrics `train_cnn_2d_tracker_marker.py` doesn't compute. Also saves an error-distribution histogram and a qualitative grid of predicted-vs-ground-truth ball points and mask contours.

In [ ]:
%cd /content/ball_balance_video_controlled
!python -m host_software.ml_vision.evaluations.evaluate_shared_vision_backbone \
    --csv-file host_software/data/03_gold/{DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{DATASET_NAME}/masks \
    --checkpoint {output_dir}/shared_vision_backbone_best.pt

In [ ]:
import json

display(Image(filename=f'{output_dir}/evaluation_error_histogram.png'))
display(Image(filename=f'{output_dir}/evaluation_visual_grid.png'))
with open(f'{output_dir}/evaluation_metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))